In [0]:
import sys, os
import pytz
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpyme/'))

# Install dependecies
packages = ['xgboost',
            'lightgbm',
            'catboost',
            'openpyx1',
            'category_encoders',            
            'fsspec']

for package in packages:
    os.system(f"pip install {package}--quiet")
from decorators import *


from sklearn.model_selection import StratifiedKFold, KFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from auxiliary_functions import spark_to_pandas

#from matplotlib_venn import venn2
from sklearn import preprocessing
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
from glob import glob
#from tqdm import tqdm
import pandas as pd
import numpy as np

%matplotlib inline
import itertools
import warnings
import os
import gc
pd.options.display.max_columns=None
warnings.filterwarnings("ignore")

from pyspark.sql.functions import col, count , date_format
from pyspark.sql import functions as F
from pyspark.sql.functions import col, countDistinct
import sys, os
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpyme/'))


from utils import *
from write_to storage import *
from write_to storage import pd_read_chunks
from pyspark.sql import SparkSession
from catboost import CatBoostClassifier, Pool

In [0]:
df = pd_read_chunks(
    directory_path="/Workspace/Users/sherlytsalazar@bcp.com.pe/02_Proyectos/06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/base_202306_202601_PD", file_format='parquet',
    verbose=True).sort_values(by=["codmes", "codclaveunicocli"], ascending=[True, True])

In [0]:
codmes_proceso = 202506  # hace equivalente al codmes data de la tabla score behavior troncal

In [0]:
df1 = df.loc[df["codmes"] == codmes_proceso, ['codmes', 'codclaveunicocli',
    'MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12',
    'MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m',
    'MOD_ACT__pctratiomtoopeaprmu6mopecprmu12',
    'MOD_DEMO__ctdmesantiguedadempsunat',
    'APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl',
    'APP_SCORE_APROB_PYME__edad_fin',
    'APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag',
    'APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl',
    'APP_SCORE_APROB_PYME__utl_3_rl',
    'APP_SCORE_APROB_PYME__act_eco_fin',
    'RNG_ACTIVIDAD_ECONOM'
]]

In [0]:
pd_202509_202601 = df.loc[df["codmes"] == codmes_proceso,
                          ['codmes', 'codclaveunicocli', 'codclavepartycli', 'codinternocomputacional']]

bd_202509_202601 = spark.createDataFrame(pd_202509_202601)

bd_202509_202601.cache()
print(f"Registros base: {bd_202509_202601.count()}")  # Fuerza el cache

In [0]:
bd_202509_202601.select("codmes").distinct().display()

##TODAS LAS VARIABLES "hm_scorepreaprobacionapppyme"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType
from pyspark.sql.functions import udf, array

codmes_data = 202512
codmes_proceso = 202601 # hace equivalente al codmes data de la tabla score behavior troncal
# ===============================================
# BASE: Universo de clientes
# ===============================================
bd_202509_202601 = (
    spark.table("catalog_lhcl_prod_bcp.bcp_edv_fabseg.BD_CLIENTE_PD_BHV_TRONCAL_202509_202601_V060226")
    .select("codmes", "codclaveunicocli", "codclavepartycli", "codinternocomputacional")
    .filter(F.col("codmes") == codmes_proceso)
)
bd_202509_202601.cache()
print(f"Registros base: {bd_202509_202601.count()}")

# ===============================================
# PASO 1: tippartyidentificacion
# ===============================================
cliente_prospecto = spark.table(
    "catalog_lhcl_prod.bcp.bcp_ddv_rbmbcapym_modelogestion_vu.hm_clienteprospectopyme"
).filter(F.col("codmes") == codmes_data)\
    select("CODCLAVEUNICOCLI","tippartyidentificacion")

# ================================
# PASO 2: Relacionados
# ================================
relacionados = spark.table(
    "catalog_lhcl_prod_bcp_bdv_rbmbcapym_apppyme_vu.hm_relacionadoapppyme"
).filter(f.col("codmes") ==codmes_data)\
 .select("CODCLAVEUNICOCLI", "CODCLAVEUNICOCLIREL", "DESTIPREL", "PCTPARTICIPACIONREL")

relacionados = relacionados.join(cliente_prospecto,    ["CODCLAVEUNICOCLI"], "left_outer")

# ================================
# PASO 3: Filtrar DUENIO/DUENIO P.N.
# ================================
duenios = relacionados\
 .filter(f.col("DESTIPREL").isin('DUENIO', 'DUENIO P.N.'))\
 .select(
    "CODCLAVEUNICOCLI",
    "CODCLAVEUNICOCLIREL",
    f.when(f.col("tippartyidentificacion") == '6', 'J')
     .otherwise('N').alias("FLGTIPPER")
)

# ================================
# PASO 4: UN SOLO dueño por cliente
# ================================
duenio_unico = duenios\
    .groupBy("CODCLAVEUNICOCLI")\
    .agg(
        F.max(
            F.when(F.col('FLGTIPPER') == 'N', F.col('CODCLAVEUNICOCLI'))\
            .otherwise(F.col('CODCLAVEUNICOCLIREL'))\
        ).alias('CODCLAVEUNICOCLIREL')
    )

# ================================
# PASO 5: Marcar FLGRLDUENIO
# ================================
relacionados_con_flag = relacionados.alias("A").join(
    duenio_unico.alias("B"),
    (F.col("A.CODCLAVEUNICOCLI") == F.col("B.CODCLAVEUNICOCLI")) &
    (F.col("A.CODCLAVEUNICOCLIREL") == F.col("B.CODCLAVEUNICOCLIREL")),
    "left_outer"
).select(
    F.col("A.CODCLAVEUNICOCLI"),
    F.col("A.CODCLAVEUNICOCLIREL"),
    F.col("A.DESTIPREL"),
    F.col("A.PCTPARTICIPACIONREL"),
    F.when(
        F.col("A.DESTIPREL").isin('DUENIO', 'DUENIO P.N.'),
        F.when(F.col("B.CODCLAVEUNICOCLI").isNull(), 0).otherwise(1)
    ).otherwise(0).alias("FLGRELDUENIO")
)

# ================================
# PASO 6: Filtrar solo el dueño seleccionado
# ================================
relacion_dueno_final = relacionados_con_flag\
    .filter(F.col('FLGRELDUENIO') == 1)\
    .select( 'CODCLAVEUNICOCLI', 'CODCLAVEUNICOCLIREL')

# ================================
# PASO 7: Campos del UNIVERSO (edad_fin, act_eco_fin)
# ================================
universo_apppyme = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_clientepreaprobacionapppyme"
).filter(F.col("codmes") == codmes_data)\
 .select(
    "CODCLAVEUNICOCLI",
    F.col("NUMEDAD").cast(IntegerType()).alias("EDAD_FIN"),
    F.col("DESSECCIONECONOMICAAPPPYME").alias("ACT_ECO_FIN"),
    "TIPPARTYIDENTIFICACION"
)

# ================================
# PASO 8: Variables de CARRETERA (hm_matrizdeudorrccproducto)
# ================================
carretera_rcc = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
).filter(F.col("codmes") == codmes_data)\
 .select(
    "codclaveunicocli",
    F.col("RCC_TIP_COND_MOR_MAX_CRNNR_MAX_U12").alias("ATRASOMAX_CRNENR_12"),
    F.col("RCC_MTO_ADE_ACT_SHIP_RT_U6M").alias("MONTOADE_ACT_MAX6_S_HIP"),
    F.col("RCC_PCT_UTL_3_RT_U3M").alias("UTL_3"),
    F.col("RCC_CTD_SF_CAL_CPP_FRQ_0_U24").alias("SF_NUM_CAL_CPP24")
)

# ============================================
# PASO 9: Variables de RESUMEN SALDO (hm_matrizresumensaldoactivopasivo)
# ============================================
resumen_saldo = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo"
).filter(F.col("codmes") == codmes_data)\
 .select(
    "codclaveunicocli",
    F.col("PROD_MTO_SLD_PRM_TSAV_FRQ_100_U24").alias("MESES_PMSAVMF_24_100"),
    F.col("PROD_CTD_MES_PAS_ACT_MAX_V2_1000_U12").alias("MESES_PASIVO_ACTIVO_12_1000")
)

# ============================================
# PASO 10: Variables de MATERIALIDAD
# ============================================
materialidad = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablerccmaterialidadapppyme"
).filter(F.col("codmes") == codmes_data)\
 .select(
    "codclaveunicocli",
    F.col("RCC_TIP_COND_MOR_MAX_CRNNR_100_MAX_U12").alias("ATRASOMAX_CRNENR_12_100"),
    F.col("RCC_CTD_SF_CAL_CPP_FRQ_100_U24").alias("SF_NUM_CAL_CPP24_100"),
    F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_100_U6M").alias("MESES_ACTIVO_SF_BU_MA6_100")
)

# ============================================
# PASO 11: Construir carretera CON materialidad (fullouter)
# ============================================
# Primero unir carretera_rcc + resumen_saldo
carretera_completa = carretera_rcc.join(    resumen_saldo, ["codclaveunicocli"], "fullouter")

# Luego unir con materialidad
carretera_con_mat = carretera_completa.join(materialidad, ["codclaveunicocli"], "fullouter")

# Aplicar lógica de materialidad: si existe _100, se usa; si no, se usa el base
carretera_final = carretera_con_mat.select(
    "codclaveunicocli",
    # ATRASOMAX_CRNENR_12: materialidad reemplaza
    F.when(F.col("ATRASOMAX_CRNENR_12_100").isNull(), F.col("ATRASOMAX_CRNENR_12"))
     .otherwise(F.col("ATRASOMAX_CRNENR_12_100")).alias("ATRASOMAX_CRNENR_12"),
    # SF_NUM_CAL_CPP24: materialidad reemplaza
    F.when(F.col("SF_NUM_CAL_CPP24_100").isNull(), F.col("SF_NUM_CAL_CPP24"))
     .otherwise(F.col("SF_NUM_CAL_CPP24_100")).alias("SF_NUM_CAL_CPP24"),
    # MESES_ACTIVO_SF_BU_MA6_0: materialidad reemplaza
    # OJO: en el código original el campo base es RCC_CTD_MES_ACT_SF_BUEN_MAL_0_UGM
    # pero en variablesMaterialidad se mapea MESES_ACTIVO_SF_BU_MA6_100 ->
    F.col("MESES_ACTIVO_SF_BU_MA6_100").alias("MESES_ACTIVO_SF_BU_MA6_0_MAT"),
    # Campos directos (sin materialidad)
    "MONTOADE_ACT_MA6_S_HIP",
    "UTL_3",
    "MESES_PMSAVMF_24_100"
)

# ================================================
# PASO 11b: Necesitamos el campo base MESES_ACTIVO_SF_BU_MA6_0 de carretera
# ================================================
carretera_meses_sf = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
).filter(f.col("codmes") == codmes_data)
.select(
    "codclaveunicocli",
    F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_0_U6M").alias("MESES_ACTIVO_SF_BU_MA6_0_BASE")
)

carretera_final = carretera_final.join(carretera_meses_sf, ["codclaveunicocli"], "left_outer")

# Aplicar materialidad para MESES_ACTIVO_SF_BU_MA6_0
carretera_final = carretera_final.withColumn(
    "MESES_ACTIVO_SF_BU_MA6_0",
    F.when(f.col("MESES_ACTIVO_SF_BU_MA6_0_MAT").isNull(), F.col("MESES_ACTIVO_SF_BU_MA6_0_BASE"))
    .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_MAT"))
).drop("MESES_ACTIVO_SF_BU_MA6_0_MAT", "MESES_ACTIVO_SF_BU_MA6_0_BASE")

# ==========================================
# PASO 12: Obtener variables del DUEÑO (_RL)
# ==========================================
variables_dueno = relacion_dueno_final.join(
    carretera_final.select(
        F.col("codclaveunicocli").alias("CODCLAVEUNICOCLIREL"),
        F.col("ATRASOMAX_CRNENR_12").alias("ATRASOMAX_CRNENR_12_RL"),
        F.col("MONTOADE_ACT_MAX6_S_HIP").alias("MONTOADE_ACT_MAX6_S_HIP_RL"),
        F.col("UTL_3").alias("UTL_3_RL"),
        F.col("SF_NUM_CAL_CPP24").alias("SF_NUM_CAL_CPP24_RL"),
        F.col("MESES_ACTIVO_SF_BU_MA6_0").alias("MESES_ACTIVO_SF_BU_MA6_0_RL"),
    ),
    "CODCLAVEUNICOCLIREL",
    "left_outer"
).select(
    "CODCLAVEUNICOCLI",
    "ATRASOMAX_CRNENR_12_RL",
    "MONTOADE_ACT_MAX6_S_HIP_RL",
    "UTL_3_RL",
    "SF_NUM_CAL_CPP24_RL",
    "MESES_ACTIVO_SF_BU_MA6_0_RL",
)

# Verificar unicidad
variables_dueno.groupBy("CODCLAVEUNICOCLI").count() \
    .filter(F.col("count") > 1).show()

# ============================================
# PASO 13: Obtener variables del CLIENTE (para AG)
# ============================================
# SF_NUM_CAL_CPP24 y MESES_ACTIVO_SF_BU_MAG_0 del propio cliente
variables_cliente = carretera_final.select(
    "codclaveunicocli",
    "SF_NUM_CAL_CPP24",
    "MESES_ACTIVO_SF_BU_MA6_0"
)

# ============================================
# PASO 14: UDF para calcular MAX entre columnas (replica del código original)
# ============================================
def operacionesMaxBetweenCols(columnas):
    resultado = 0.0
    valorInicial = float(-999999999999999.0)
    mayor = float(0.0)
    numero = float(0.0)
    for column in columnas:
        if column is not None and str(column) != "" and str(column).upper() != "NULL":
            numero = float(column)
        else:
            numero = -999999999999999.0
        if numero >= valorInicial:
            mayor = float(numero)
            valorInicial = float(numero)
        elif numero < valorInicial:
            mayor = float(valorInicial)
        if mayor == -999999999999999.0:
            resultado = None
        else:
            resultado = mayor
    return resultado

operacionesMaxBetweenCols_udf = udf(operacionesMaxBetweenCols, DoubleType())

# ================================
# PASO 15: Unir todo y calcular campos AG
# ================================
resultado = bd_202509_202601 \
    .join(universo_apppyme, ["CODCLAVEUNICOCLI"], "left_outer") \
    .join(variables_dueno, ["CODCLAVEUNICOCLI"], "left_outer") \
    .join(variables_cliente, ["CODCLAVEUNICOCLI"], "left_outer")

# --- MESES_ACTIVO_SF_BU_MA6_0_AG ---
# Misma lógica: PN -> cliente, PJ -> max(cliente, dueño)
resultado = resultado.withColumn(
    "MESES_ACTIVO_SF_BU_MA6_0_AG_raw",
    operacionesMaxBetweenCols_udf(
        array(F.col("MESES_ACTIVO_SF_BU_MA6_0"), F.col("MESES_ACTIVO_SF_BU_MA6_0_RL"))
    )
).withColumn(
    "MESES_ACTIVO_SF_BU_MA6_0_AG",
    F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("MESES_ACTIVO_SF_BU_MA6_0"))
     .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_AG_raw"))
     .cast(IntegerType())
)

# ============================================
# PASO 16: Castear campos _RL a tipos correctos
# ============================================
resultado = resultado.select(
    "codmes",
    "codclaveunicocli",
    "codclavepartycli",
    "codinternocomputacional",
    # Campos del universo
    F.col("EDAD_FIN").cast(IntegerType()).alias("edad_fin"),
    F.col("ACT_ECO_FIN").alias("act_eco_fin"),
    # Campos _RL (del dueño)
    F.col("ATRASOMAX_CRNENR_12_RL").cast(IntegerType()).alias("atrasomax_crnenr_12_rl"),
    F.col("MONTOADE_ACT_MAX6_S_HIP_RL").cast(DecimalType(precision=19, scale=8)).alias("montoade_act_max6_s_hip_rl"),
    F.col("UTL_3_RL").cast(DecimalType(precision=23, scale=6)).alias("utl_3_rl"),
    # Campos _AG (agregados)
    F.col("MESES_ACTIVO_SF_BU_MA6_0_AG").cast(IntegerType()).alias("meses_activo_sf_bu_ma6_0_ag")
)

# ============================================
# PASO 17: Limpieza de Datos
# ============================================
GLOB_DUMMY_LIST = [1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -3333333333, 4444444444, 5555555555, 6666666666, 7777777777, -99, -999,
                   4444444444, 5555555555, 6666666666, 7777777777, 1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -99, -3333333333, 44444.4444,
                   555555.5555, 666666.6666, 77777.7777, 111111.1111, -111111.1111, 222222.2222, -222222.2222, 333333.3333, -333333.3333, None]

cols = resultado.columns
resultado = (resultado.select(*[F.when(F.col(c).isin(GLOB_DUMMY_LIST), None)\
                        .otherwise(F.col(c)).alias(c) for c in cols]))

# ============================================
# PASO 18: Verificar resultados
# ============================================
resultado.display()
print(f"Total registros: {resultado.count()}")

In [0]:
# Renombrar columnas del pandas

df_pandas_renamed = df1.rename(columns={
    'APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl': 'atrasomax_crnenr_12_rl_pandas',
    'APP_SCORE_APROB_PYME__edad_fin': 'edad_fin_pandas',
    'APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag': 'meses_activo_sf_bu_ma6_0_ag_pandas',
    'APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl': 'montoade_act_max6_s_hip_rl_pandas',
    'APP_SCORE_APROB_PYME__utl_3_rl': 'utl_3_rl_pandas',
    'APP_SCORE_APROB_PYME__act_eco_fin': 'act_eco_fin_pandas'
})

spark_from_pandas = spark.createDataFrame(
    df_pandas_renamed[[
        'codclaveunicocli',
        'atrasomax_crnenr_12_rl_pandas',
        'edad_fin_pandas',
        'meses_activo_sf_bu_ma6_0_ag_pandas',
        'montoade_act_max6_s_hip_rl_pandas',
        'utl_3_rl_pandas',
        'act_eco_fin_pandas'
    ]]
)

# Join
comparacion = resultado.join(spark_from_pandas, on="codclaveunicocli", how="inner")

# Función helper para comparar incluyendo nulls
def col_equal_with_nulls(col1, col2):
    return (
        (F.col(col1) == F.col(col2)) |  # Ambos tienen valor y son iguales
        (F.col(col1).isNull() & F.col(col2).isNull())  # Ambos son null
    )

# Agregar flags de coincidencia (null == null -> True)
comparacion = comparacion\
    .withColumn("ok_atrasomax", col_equal_with_nulls("atrasomax_crnenr_12_rl", "atrasomax_crnenr_12_rl_pandas"))\
    .withColumn("ok_edad", col_equal_with_nulls("edad_fin", "edad_fin_pandas"))\
    .withColumn("ok_meses_sf", col_equal_with_nulls("meses_activo_sf_bu_ma6_0_ag", "meses_activo_sf_bu_ma6_0_ag_pandas"))\
    .withColumn("ok_montoade", col_equal_with_nulls("montoade_act_max6_s_hip_rl", "montoade_act_max6_s_hip_rl_pandas"))\
    .withColumn("ok_utl3", col_equal_with_nulls("utl_3_rl", "utl_3_rl_pandas"))\
    .withColumn("ok_acteco", col_equal_with_nulls("act_eco_fin", "act_eco_fin_pandas"))

# Resumen
resumen = comparacion.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("ok_atrasomax") == True, 1).otherwise(0)).alias("atrasomax_OK"),
    F.sum(F.when(F.col("ok_edad") == True, 1).otherwise(0)).alias("edad_OK"),
    F.sum(F.when(F.col("ok_meses_sf") == True, 1).otherwise(0)).alias("meses_sf_OK"),
    F.sum(F.when(F.col("ok_pmsavmf") == True, 1).otherwise(0)).alias("pmsavmf_OK"),
    F.sum(F.when(F.col("ok_montoade") == True, 1).otherwise(0)).alias("montoade_OK"),
    F.sum(F.when(F.col("ok_sfnum") == True, 1).otherwise(0)).alias("sfnum_OK"),
    F.sum(F.when(F.col("ok_utl3") == True, 1).otherwise(0)).alias("utl3_OK"),
    F.sum(F.when(F.col("ok_acteco") == True, 1).otherwise(0)).alias("acteco_OK"),
    # NO coinciden
    F.sum(F.when(F.col("ok_atrasomax") == False, 1).otherwise(0)).alias("atrasomax_NO"),
    F.sum(F.when(F.col("ok_edad") == False, 1).otherwise(0)).alias("edad_NO"),
    F.sum(F.when(F.col("ok_meses_sf") == False, 1).otherwise(0)).alias("meses_sf_NO"),
    F.sum(F.when(F.col("ok_pmsavmf") == False, 1).otherwise(0)).alias("pmsavmf_NO"),
    F.sum(F.when(F.col("ok_montoade") == False, 1).otherwise(0)).alias("montoade_NO"),
    F.sum(F.when(F.col("ok_sfnum") == False, 1).otherwise(0)).alias("sfnum_NO"),
    F.sum(F.when(F.col("ok_utl3") == False, 1).otherwise(0)).alias("utl3_NO"),
    F.sum(F.when(F.col("ok_acteco") == False, 1).otherwise(0)).alias("acteco_NO")
)

resumen.display()

# Ver detalle de los que NO coinciden
no_coinciden = comparacion.filter(
    (F.col("ok_atrasomax") == False) |
    (F.col("ok_edad") == False) |
    (F.col("ok_meses_sf") == False) |
    (F.col("ok_pmsavmf") == False) |
    (F.col("ok_montoade") == False) |
    (F.col("ok_sfnum") == False) |
    (F.col("ok_utl3") == False) |
    (F.col("ok_acteco") == False)
)

no_coinciden.display()

##TODAS LAS VARIABLES "hm_scoreappbasepymemodulodemografico"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

codmes_data = 202512
codmes_proceso= 202601

# ================================================
# PASO 1: Obtener ctdmesantiguedadempsunat
# -----------------------------------------------
contribuyente_sunat = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_modelogestion_vu.hm_contribuyentesunatpyme"
).filter(F.col("CODMES") == codmes_data)\
    .select(
        F.col("CODCLAVEUNICOCLI"),
        F.col("CTDMESANTIGUEDADEMP").cast(IntegerType()).alias("ctdmesantiguedadempsunat")
)

# ================================================
# PASO 2: Unir al universo principal
# ================================================
resultado = bd_202509_202601.join(
    contribuyente_sunat,
    "CODCLAVEUNICOCLI",
    "left_outer"
)

In [0]:
# Renombrar columna del pandas
df_pandas_renamed = df1.rename(columns={
    'MOD_DEMO__ctdmesantiguedadempsunat': 'ctdmesantiguedadempsunat_pandas'
})

spark_from_pandas = spark.createDataFrame(
    df_pandas_renamed[['codclaveunicocli', 'ctdmesantiguedadempsunat_pandas']]
)

# Join
comparacion = resultado.select(
    "codclaveunicocli",
    F.col("ctdmesantiguedadempsunat").alias("ctdmesantiguedadempsunat_spark")
).join(spark_from_pandas, on="codclaveunicocli", how="inner")

# Flag de coincidencia (incluyendo null == null)
comparacion = comparacion.withColumn(
    "ok_sunat",
    F.col("ctdmesantiguedadempsunat_spark").eqNullSafe(F.col("ctdmesantiguedadempsunat_pandas"))
)

# Resumen
resumen = comparacion.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("ok_sunat") == True, 1).otherwise(0)).alias("sunat_OK"),
    F.sum(F.when(F.col("ok_sunat") == False, 1).otherwise(0)).alias("sunat_NO")
)

resumen.display()

# Detalle de no coincidentes
no_coinciden = comparacion.filter(F.col("ok_sunat") == False)
no_coinciden.display()

##TODAS LAS VARIABLES “hm_scoreappbasepymemoduloactivo”

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.window import Window
from dateutil.relativedelta import relativedelta
from datetime import datetime

# ==================================================
# PARAMETROS
# ==================================================

codmescuatro = 202502
codmestres = 202503
codmes_data = 202505
codmes_proceso = 202506

def generar_meses(mes_inicio, mes_fin):
    meses = []
    fecha_fin = datetime.strptime(str(mes_fin), "%Y%m")
    fecha_actual = datetime.strptime(str(mes_inicio), "%Y%m")
    while fecha_actual <= fecha_fin:
        meses.append(fecha_actual.strftime("%Y%m"))
        fecha_actual += relativedelta(months=1)
    return meses

# ================================
# UNIVERSO MÓDULO ACTIVO (filtro clave)
# Solo clientes con flgactmay100may6u12 = 1
# ================================
universo_segmento = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_segmentacionpyme_vu.hm_segmentoinformativopyme"
).filter(
    (F.col("codmes") == codmes_data) &
    (F.col("flgactmay100may6u12") == 1)
).select("CODCLAVEUNICOCLI").distinct()

# ================================
# CAMPO 1 y 3: Transacciones
# ================================
transacciones = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_matrizbasepasivoclienteapppyme"
).filter(F.col("codmes") == codmes_data)
.select(
    F.col("CODCLAVEPARTYCLI"),
    (F.col("ISAV_MTO_OPEA_ESTVTA_PYM_PRM_U6M") / F.col("ISAV_MTO_OPEA_ESTVTA_PYM_MAX_U12"))
        .alias("isav_mto_opea_estvta_pym_u6m_rt_max_u12"),
    (F.col("ISAV_MTO_OPEA_ESTVTA_PYM_PRM_U6M") / F.col("ISAV_MTO_OPEC_ESTVTA_PYM_PRM_U12"))
        .alias("pctratiomtoopeaprmu6mopecprmu12")
)

# ==================================================
# CAMPO 2: Variación de deuda / Pasivos
# ==================================================

# Portafolio
portafolio = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito"
).filter(
    (F.col("CODMES").between(codmescuatro, codmes_data)) &
    (F.trim(F.col("FLGCTAVALIDA")) == "1") &
    (F.col("TIPESTADOCTA").isin("A", "AC", "D"))
).select("CODCLAVEPARTYCLI", "MTOSALDOCAPITALSOL", "CODMES")

portafolio_agrupado = portafolio.groupBy("CODCLAVEPARTYCLI", "CODMES").agg(
    F.sum("MTOSALDOCAPITALSOL").alias("mtosaldocapitalsol_tmp")
)

# CrossJoin con todos los meses
lista_meses = generar_meses(codmescuatro, codmes_data)
df_meses = spark.createDataFrame(lista_meses, StringType()).select(F.col("value").alias("CODMES"))

partys = portafolio_agrupado.select("CODCLAVEPARTYCLI").dropDuplicates()
cross = partys.crossJoin(F.broadcast(df_meses))
portafolio_completo = portafolio_agrupado.join(cross, on=["CODMES", "CODCLAVEPARTYCLI"], how="right_outer")

# Variación
w = Window.partitionBy("CODCLAVEPARTYCLI").orderBy(F.col("CODMES").desc())

portafolio_var = portafolio_completo.withColumn(
    "mtosaldocapitalsol_tmp_prev",
    F.lead("mtosaldocapitalsol_tmp", 1).over(w).cast("double")
).withColumn(
    "mtosaldocapitalsol_tmp_VAR_DEC",
    F.when(
        F.col("mtosaldocapitalsol_tmp").isNull() & F.col("mtosaldocapitalsol_tmp_prev").isNull(),
        F.lit(None)
    ).otherwise(
        F.when(
            F.coalesce(F.col("mtosaldocapitalsol_tmp"), F.lit(0)) -
            F.coalesce(F.col("mtosaldocapitalsol_tmp_prev"), F.lit(0)) > 0,
            F.lit(0)
        ).otherwise(
            F.abs(
                F.coalesce(F.col("mtosaldocapitalsol_tmp"), F.lit(0)) -
                F.coalesce(F.col("mtosaldocapitalsol_tmp_prev"), F.lit(0))
            )
        )
    )
)

dec_var_prm3 = portafolio_var.filter(
    F.col("CODMES").between(str(codmes_tres), str(codmes_data))
).groupBy("CODCLAVEPARTYCLI").agg(
    F.avg("mtosaldocapitalsol_tmp_VAR_DEC").alias("dec_var_MONTO_meanPrev3")
)

# Pasivos
pasivos = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_matrizbaseresumensaldoapppyme"
).filter(F.col("codmes") == codmes_data)\
.select("CODCLAVEUNICOCLI", "PROD_MTO_SLD_PRM_VIG_PAS_AHO_CTEORD_PRM_U3M")

# ================================
# UNIR TODO CON FILTRO DE UNIVERSO 
# ================================

# Marcar quiénes están en el universo del módulo activo
bd_con_flag = bd_202509_202601.join(
    universo_segmento.withColumn("en_universo", F.lit(1)),
    "CODCLAVEUNICOCLI",
    "left"
)

# Joins de insumos
resultado = bd_con_flag\
    .join(transacciones, "CODCLAVEPARTYCLI", "left")\
    .join(dec_var_prm3, "CODCLAVEPARTYCLI", "left")\
    .join(pasivos, "CODCLAVEUNICOCLI", "left")

# Calcular campo 2 intermedio
resultado = resultado.withColumn(
    "pctratiomtodecdeudapymertmtopasivoprmu3m_raw",
    F.col("dec_var_MONTO_meanPrev3") / F.col("PROD_MTO_SLD_PRM_VIG_PAS_AHO_CTEORD_PRM_U3M")
)

# ================================
# APLICAR REGLA: Si NO está en universo -> NULL
# ================================

resultado = resultado.withColumn(
    "isav_mto_opea_estvta_pym_u6m_rt_max_u12",
    F.when(F.col("en_universo") == 1, F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12"))
     .otherwise(F.lit(None))
).withColumn(
    "pctratiomtodecdeudapymertmtopasivoprmu3m",
    F.when(F.col("en_universo") == 1, F.col("pctratiomtodecdeudapymertmtopasivoprmu3m_raw"))
     .otherwise(F.lit(None))
).withColumn(
    "pctratiomtoopeaprmu6mopecprmu12",
    F.when(F.col("en_universo") == 1, F.col("pctratiomtoopeaprmu6mopecprmu12"))
     .otherwise(F.lit(None))
)

# ================================
# Selección final con cast
# ================================
resultado = resultado.select(
    "codmes",
    "codclaveunicocli",
    F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12").cast(DecimalType(19, 8)).alias("isav_mto_opea_estvta_pym_u6m_rt_max_u12"),
    F.col("pctratiomtodecdeudapymertmtopasivoprmu3m").cast(DecimalType(19, 8)).alias("pctratiomtodecdeudapymertmtopasivoprmu3m"),
    F.col("pctratiomtoopeaprmu6mopecprmu12").cast(DecimalType(19, 8)).alias("pctratiomtoopeaprmu6mopecprmu12")
)

# ================================
# Limpieza de dummies
# ================================
lista_dummies = [1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -3333333333, 4444444444, 5555555555, 6666666666, 7777777777, -99, -999,
                   4444444444, 5555555555, 6666666666, 7777777777, 1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -99, -3333333333, 44444.4444,
                   555555.5555, 666666.6666, 77777.7777, 111111.1111, -111111.1111, 222222.2222, -222222.2222, 333333.3333, -333333.3333, None]

cols = resultado.columns
resultado = (resultado.select(*[F.when(F.col(c).isin(lista_dummies), None)\
                        .otherwise(F.col(c)).alias(c) for c in cols]))

In [0]:
bd_202509_202601.limit(2).display()

In [0]:
# ========================================
# COMPARACIÓN
# ========================================

# Renombrar columnas del pandas
df_pandas_renamed = df1.rename(
    columns={
        'MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12': 'isav_mto_opea_estvta_pym_u6m_rt_max_u12_pandas',
        'MOD_ACT__pctratiomtodecdeudapymer tmtopasivoprmu3m': 'pctratiomtodecdeudapymer tmtopasivoprmu3m_pandas',
        'MOD_ACT__pctratiomtoopeaprmu6mopecprmu12': 'pctratiomtoopeaprmu6mopecprmu12_pandas'
    }
)

spark_from_pandas = spark.createDataFrame(
    df_pandas_renamed[[
        'codclaveunicocli',
        'isav_mto_opea_estvta_pym_u6m_rt_max_u12_pandas',
        'pctratiomtodecdeudapymer tmtopasivoprmu3m_pandas',
        'pctratiomtoopeaprmu6mopecprmu12_pandas'
    ]]
)

# Join
comparacion = resultado.join(spark_from_pandas, on="codclaveunicocli", how="inner")

def col_equal_with_nulls(col1, col2):
    return F.when(
        F.col(col1).isNull() & F.col(col2).isNull(), True  # Ambos NULL -> iguales
    ).when(
        F.col(col1).isNull() | F.col(col2).isNull(), False  # Solo uno NULL -> distintos
    ).otherwise(
        F.col(col1) == F.col(col2)  # Ambos con valor -> comparar
    )

# Agregar flags de coincidencia
comparacion = comparacion\
    .withColumn("ok_opea_rt_max", col_equal_with_nulls("isav_mto_opea_estvta_pym_u6m_rt_max_u12",           "isav_mto_opea_estvta_pym_u6m_rt_max_u12_pandas"))\
    .withColumn("ok_decdeuda_pasivo", col_equal_with_nulls("pctratiomtodecdeudapymertmtopasivoprmu3m", "pctratiomtodecdeudapymertmtopasivoprmu3m_pandas"))\
    .withColumn("ok_opea_opec", col_equal_with_nulls("pctratiomtoopeaprmu6mopecprmu12", "pctratiomtoopeaprmu6mopecprmu12_pandas"))

# Resumen
resumen = comparacion.agg(
    F.count("*").alias("total"),
    # OK
    F.sum(F.when(F.col("ok_opea_rt_max") == True, 1).otherwise(0)).alias("opea_rt_max_OK"),
    F.sum(F.when(F.col("ok_decdeuda_pasivo") == True, 1).otherwise(0)).alias("decdeuda_pasivo_OK"),
    F.sum(F.when(F.col("ok_opea_opec") == True, 1).otherwise(0)).alias("opea_opec_OK"),
    # NO
    F.sum(F.when(F.col("ok_opea_rt_max") == False, 1).otherwise(0)).alias("opea_rt_max_NO"),
    F.sum(F.when(F.col("ok_decdeuda_pasivo") == False, 1).otherwise(0)).alias("decdeuda_pasivo_NO"),
    F.sum(F.when(F.col("ok_opea_opec") == False, 1).otherwise(0)).alias("opea_opec_NO")
)

resumen.display()

# Ver detalle de los que NO coinciden
no_coinciden = comparacion.filter(
    (F.col("ok_opea_rt_max") == False) |
    (F.col("ok_decdeuda_pasivo") == False) |
    (F.col("ok_opea_opec") == False)
).select(
    "codclaveunicocli",
    # Campo 1
    "isav_mto_opea_estvta_pym_u6m_rt_max_u12",
    "isav_mto_opea_estvta_pym_u6m_rt_max_u12_pandas",
    "ok_opea_rt_max",
    # Campo 2
    "pctratiomtodecdeudapymertmtopasivoprmu3m",
    "pctratiomtodecdeudapymertmtopasivoprmu3m_pandas",
    "ok_decdeuda_pasivo",
    # Campo 3
    "pctratiomtoopeaprmu6mopecprmu12",
    "pctratiomtoopeaprmu6mopecprmu12_pandas",
    "ok_opea_opec"
)

no_coinciden.display()